# Particle Filter — Ball Tracking (THWS Portfolio Exam 2, Task P2.1)

Static, plot-based analysis of the particle filter. The filter logic lives in `filter.py`,
the simulated world in `simulation.py`, and `experiment.py` runs headless episodes —
no pygame needed here. (For the *interactive* visualization run `python viz_pygame.py`.)

See `README.md` for design rationale and `CONCEPT.md` for the conceptual overview.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from experiment import run_episode

COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:pink", "tab:olive",
          "tab:purple", "tab:cyan", "tab:red"]

# ---- parameters (tweak freely and re-run the notebook) ----
PARAMS = dict(
    n_balls=3,          # number of simultaneously flying balls
    n_particles=4000,   # particle count
    obs_noise=3.0,      # sensor noise std dev [m]
    obs_interval=0.15,  # mean time between sensor reports [s] (jittered ±30 %)
    dropout_prob=0.02,  # chance per report that a complete dropout starts
    dropout_len=1.5,    # dropout duration [s]
    launch_area=50.0,   # unknown 50×50 m launch region
    relaunch=False,     # True: balls relaunch after landing
    duration=10.0,      # simulated seconds
    seed=7,             # reproducibility
)

hist = run_episode(**PARAMS)
errs = [e for e in hist["err"] if e is not None]
print(f"frames: {len(hist['t'])}, mean position error: {np.mean(errs):.2f} m")

## 1. Trajectories: truth vs. observations vs. estimates

Black lines are the true flight parabolas, gold dots the noisy unlabeled sensor reports,
and the colored dots the filter's position estimates over time (one color per tracked mode).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ox = [p[0] for o in hist["obs"] if o for p in o]
oy = [p[1] for o in hist["obs"] if o for p in o]
ax.scatter(ox, oy, s=10, c="gold", alpha=0.4, label="noisy observations")
n_balls = len(hist["truth"][0])
for b in range(n_balls):
    xs = [fr[b][0] for fr in hist["truth"] if fr[b][4]]
    ys = [fr[b][1] for fr in hist["truth"] if fr[b][4]]
    ax.plot(xs, ys, "k-", lw=1.5, label="true trajectory" if b == 0 else None)
for est in hist["est"]:
    for j, e in enumerate(est):
        ax.plot(e[0], e[1], ".", ms=2, c=COLORS[j % len(COLORS)])
ax.axhline(0, c="gray", lw=1)
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
ax.set_title("True trajectories (black), observations (gold), estimates (colored)")
ax.legend(); ax.set_ylim(bottom=-5)
plt.show()

## 2. The particle cloud over time

Snapshots of all particles (colored by k-means cluster), with the true ball positions as
black stars. Watch the initial uniform fog over the launch area collapse onto the balls,
and modes disappear as balls land.

In [ ]:
idx = np.linspace(0, len(hist["snap_t"]) - 1, 6).astype(int)
dt = hist["t"][1] - hist["t"][0]
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True, sharey=True)
for ax, i in zip(axes.flat, idx):
    P, lab, ts = hist["snap_P"][i], hist["snap_labels"][i], hist["snap_t"][i]
    ax.scatter(P[:, 0], P[:, 1], s=1,
               c=[COLORS[l % len(COLORS)] for l in lab], alpha=0.25)
    fi = min(int(round(ts / dt)) - 1, len(hist["truth"]) - 1)
    for b in hist["truth"][fi]:
        if b[4]:
            ax.plot(b[0], b[1], "k*", ms=12)
    ax.axhline(0, c="gray", lw=1)
    ax.set_title(f"t = {ts:.1f} s")
    ax.set_xlim(-70, 170); ax.set_ylim(-5, 100)
fig.suptitle("Particle cloud over time (stars = true balls)")
fig.tight_layout()
plt.show()

## 3. Estimation error, sensor dropouts and tracked mode count

Red bands mark complete sensor failures — the filter bridges them by pure prediction,
so the error grows during a dropout and snaps back once measurements return. The lower
panel shows how many modes the filter believes are airborne (inferred from the
measurement count, never from ground truth).

In [ ]:
t = np.array(hist["t"])
err = np.array([e if e is not None else np.nan for e in hist["err"]], float)
drop = np.array(hist["dropout"])
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True,
                               height_ratios=[3, 1])
ax1.plot(t, err, lw=1)
ax1.fill_between(t, 0, np.nanmax(err) * 1.05, where=drop, color="red",
                 alpha=0.15, label="sensor dropout")
ax1.set_ylabel("mean position error [m]"); ax1.legend()
ax1.set_title("Estimation error over time")
ax2.step(t, hist["k"], where="post")
ax2.set_ylabel("tracked"); ax2.set_xlabel("t [s]")
fig.tight_layout()
plt.show()

## 4. Velocity estimation

Velocity is never measured — it becomes observable only through consecutive positions
and the physics model. Compare estimated and true velocity of the matched (nearest) ball.

In [ ]:
verr, vt = [], []
for f, est in enumerate(hist["est"]):
    air = [b for b in hist["truth"][f] if b[4]]
    if len(est) == 0 or not air:
        continue
    for e in est:
        b = min(air, key=lambda b: (b[0]-e[0])**2 + (b[1]-e[1])**2)
        verr.append(np.hypot(e[2]-b[2], e[3]-b[3]))
        vt.append(hist["t"][f])
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(vt, verr, ".", ms=2)
ax.set_xlabel("t [s]"); ax.set_ylabel("velocity error [m/s]")
ax.set_title("Velocity estimation error (converges as positions accumulate)")
plt.show()

## 5. Parameter playground

Stress test: doubled sensor noise, sparse reports, frequent long dropouts, 4 balls.
Change anything and re-run.

In [ ]:
h2 = run_episode(n_balls=4, obs_noise=6.0, obs_interval=0.3,
                 dropout_prob=0.06, dropout_len=2.0, duration=10.0, seed=11)
t2 = np.array(h2["t"])
e2 = np.array([e if e is not None else np.nan for e in h2["err"]], float)
d2 = np.array(h2["dropout"])
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(t2, e2, lw=1)
ax.fill_between(t2, 0, np.nanmax(e2) * 1.05, where=d2, color="red", alpha=0.15)
ax.set_xlabel("t [s]"); ax.set_ylabel("error [m]")
ax.set_title("Stress test: sigma=6 m, sparse reports, heavy dropouts, 4 balls")
print(f"mean error: {np.nanmean(e2):.2f} m")
plt.show()

## 6. Interactive pygame visualization

The animated, interactive version (pause, force dropouts, change noise live, …) uses the
exact same `simulation.py`/`filter.py` modules. Run it from a terminal — or uncomment below:

In [ ]:
# !python viz_pygame.py --balls 3 --seed 7
# !python viz_pygame.py --relaunch          # endless mode with re-acquisition